# Kaggle-Ready F5-TTS Mamba3 Training (Multi-GPU T4x2)

Notebook ini disiapkan untuk:
- training F5-TTS dengan backbone `Mamba3Backbone`
- dataset format `metadata.csv + wavs/`
- integrasi `WANDB_API_KEY` dari Kaggle Secret
- multi-GPU training (2x T4) via `accelerate`

Catatan:
- Notebook **tidak** memaksa reinstall PyTorch, supaya tetap kompatibel dengan runtime CUDA bawaan Kaggle (termasuk jika runtime bergeser ke CUDA 13.x).
- Jika internet dibatasi, training tetap jalan karena sample logging otomatis dimatikan saat vocoder download gagal.


## Step 1 - Runtime paths and project settings

Notebook ini disetel untuk:
- source repo code dari dataset repo F5-TTS kamu
- source raw data **Bahasa Indonesia** dari:
  `/kaggle/input/datasets/anekazek/indonesian-voice-dataset/datasetku`
- format raw data: `metadata.csv + wavs/`

Project akan di-copy ke `/kaggle/working/F5-TTS` (writeable) sebelum training.


In [ ]:
from __future__ import annotations

import csv
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_INPUT = Path('/kaggle/input')

# Repo source (project code)
SOURCE_DATASET_CANDIDATES = [
    Path('/kaggle/input/datasets/benedictusryugunawan/f5-tts-mamba-tts-repo'),
    Path('/kaggle/input/f5-tts-mamba-tts-repo'),
    Path('/kaggle/input/f5-mamba-dit-tts'),
]

# Raw training data source
RAW_DATASET_CANDIDATES = [
    Path('/kaggle/input/datasets/anekazek/indonesian-voice-dataset/datasetku'),
    Path('/kaggle/input/indonesian-voice-dataset/datasetku'),
    Path('/kaggle/input/datasetku'),
]

# Metadata filename candidates — IndSP first, then KCV, then legacy typos
METADATA_CANDIDATES = ['metadata_indsp.csv', 'metadata.csv', 'metda_data.csv', 'meta_data.csv']

# Wav directory candidates — IndSP first, then KCV
WAVS_DIR_CANDIDATES = ['indsp', 'wavs']


def find_repo_root(base: Path) -> Path | None:
    if not base.exists():
        return None
    if (base / 'pyproject.toml').exists() and (base / 'src' / 'f5_tts').exists():
        return base
    for child in base.iterdir():
        if child.is_dir() and (child / 'pyproject.toml').exists() and (child / 'src' / 'f5_tts').exists():
            return child
    return None


def search_repo_under_kaggle_input() -> Path | None:
    if not KAGGLE_INPUT.exists():
        return None
    for pyproject in KAGGLE_INPUT.rglob('pyproject.toml'):
        repo = pyproject.parent
        if (repo / 'src' / 'f5_tts').exists():
            return repo
    return None


def pick_metadata_file(data_root: Path) -> Path | None:
    for name in METADATA_CANDIDATES:
        candidate = data_root / name
        if candidate.exists():
            return candidate
    return None


def validate_raw_layout(data_root: Path) -> tuple[Path | None, Path | None]:
    metadata = pick_metadata_file(data_root)
    if metadata is None:
        return None, None
    for wav_dir_name in WAVS_DIR_CANDIDATES:
        wav_dir = data_root / wav_dir_name
        if wav_dir.exists():
            return metadata, wav_dir
    return None, None


def resolve_raw_data_root() -> Path | None:
    for candidate in RAW_DATASET_CANDIDATES:
        meta, wavs = validate_raw_layout(candidate)
        if meta is not None and wavs is not None:
            return candidate

    # Fallback: scan /kaggle/input for any supported wav dir
    if KAGGLE_INPUT.exists():
        for wav_dir_name in WAVS_DIR_CANDIDATES:
            for p in KAGGLE_INPUT.rglob(wav_dir_name):
                root = p.parent
                meta, wavs = validate_raw_layout(root)
                if meta is not None and wavs is not None:
                    return root

    return None


SOURCE_REPO_ROOT = None
for candidate in SOURCE_DATASET_CANDIDATES:
    found = find_repo_root(candidate)
    if found is not None:
        SOURCE_REPO_ROOT = found
        break

if SOURCE_REPO_ROOT is None:
    SOURCE_REPO_ROOT = search_repo_under_kaggle_input()

RAW_DATA_ROOT = resolve_raw_data_root()

# Kita kerja di /kaggle/working karena /kaggle/input read-only
REPO_ROOT = KAGGLE_WORKING / 'F5-TTS'

DATA_ROOT = RAW_DATA_ROOT  # akan dipakai untuk metadata + wavs mentah
RAW_METADATA = None
RAW_WAV_DIR = None

if RAW_DATA_ROOT is not None:
    RAW_METADATA, RAW_WAV_DIR = validate_raw_layout(RAW_DATA_ROOT)

DATASET_NAME = 'indonesian_voice'
TOKENIZER = 'char'
PREPARED_DATA_DIR = REPO_ROOT / 'data' / f'{DATASET_NAME}_{TOKENIZER}'
ABS_METADATA = REPO_ROOT / 'data' / 'metadata_abs.csv'

print('SOURCE_REPO_ROOT    :', SOURCE_REPO_ROOT)
print('RAW_DATA_ROOT       :', RAW_DATA_ROOT)
print('RAW_METADATA        :', RAW_METADATA)
print('RAW_WAV_DIR         :', RAW_WAV_DIR)
print('REPO_ROOT (working) :', REPO_ROOT)
print('METADATA candidates :', METADATA_CANDIDATES)
print('WAVS dir candidates :', WAVS_DIR_CANDIDATES)
print('OUTPUT DATASET      :', PREPARED_DATA_DIR)


In [ ]:
if SOURCE_REPO_ROOT is None:
    raise FileNotFoundError(
        'Source repo tidak ditemukan di /kaggle/input. '
        'Pastikan dataset repo kamu ter-mount dan berisi pyproject.toml + src/f5_tts.'
    )

if not REPO_ROOT.exists():
    shutil.copytree(SOURCE_REPO_ROOT, REPO_ROOT)
    print('Repo copied to writable path:', REPO_ROOT)
else:
    print('Repo already available at:', REPO_ROOT)

if not (REPO_ROOT / 'pyproject.toml').exists():
    raise FileNotFoundError(
        f'pyproject.toml tidak ditemukan di {REPO_ROOT}. Pastikan ini root project F5-TTS.'
    )

# Fallback: cari dataset di dalam repo itu sendiri jika tidak ditemukan di /kaggle/input
if DATA_ROOT is None:
    candidate_roots = [
        REPO_ROOT / 'data',
        SOURCE_REPO_ROOT / 'data',
    ]
    for root in candidate_roots:
        meta, wavs = validate_raw_layout(root)
        if meta is not None and wavs is not None:
            DATA_ROOT = root
            RAW_METADATA = meta
            RAW_WAV_DIR = wavs
            break

if DATA_ROOT is None or RAW_METADATA is None or RAW_WAV_DIR is None:
    raise FileNotFoundError(
        'Raw dataset tidak ditemukan. Pastikan path berikut berisi metadata + indsp/ atau wavs/:\n'
        '- /kaggle/input/datasets/anekazek/indonesian-voice-dataset/datasetku\n'
        '- /kaggle/input/indonesian-voice-dataset/datasetku'
    )

os.chdir(REPO_ROOT)
print('Working directory:', Path.cwd())
print('DATA_ROOT    :', DATA_ROOT)
print('RAW_METADATA :', RAW_METADATA)
print('RAW_WAV_DIR  :', RAW_WAV_DIR)


## Step 2 - Install dependencies (safe for Kaggle CUDA runtime)

Kita install dependency training penting, tanpa memaksa reinstall `torch`.


In [ ]:
def run(cmd: list[str], cwd: Path | None = None, extra_env: dict[str, str] | None = None) -> None:
    print('+', ' '.join(shlex.quote(c) for c in cmd))
    env = os.environ.copy()
    if extra_env:
        env.update(extra_env)
    subprocess.run(cmd, check=True, cwd=str(cwd or REPO_ROOT), env=env)

def run_allow_fail(cmd: list[str], cwd: Path | None = None, extra_env: dict[str, str] | None = None) -> bool:
    print('+', ' '.join(shlex.quote(c) for c in cmd))
    env = os.environ.copy()
    if extra_env:
        env.update(extra_env)
    result = subprocess.run(cmd, check=False, cwd=str(cwd or REPO_ROOT), env=env)
    return result.returncode == 0

# Memastikan wheel agar tidak build dari source
run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'wheel', 'packaging'])

deps = [
    'accelerate>=1.13.0',
    'datasets',
    'hydra-core>=1.3.0',
    'safetensors',
    'cached_path',
    'pydub',
    'soundfile',
    'librosa',
    'pypinyin',
    'rjieba',
    'ema_pytorch>=0.5.2',
    'torchdiffeq',
    'transformers',
    'transformers_stream_generator',
    'vocos',
    'x_transformers>=1.31.14',
    'wandb',
    'click',
    'tqdm',
    # FIX #1: Mamba-3 kernel dependencies
]

print("Installing generic dependencies...")
run([sys.executable, '-m', 'pip', 'install', '-q', *deps])

print("Installing causal_conv1d and mamba_ssm (wheel-first, source fallback)...")
# Faster path on Kaggle: use prebuilt wheels first, compile from source only if needed.
wheel_ok = run_allow_fail([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--only-binary=:all:',
    'causal_conv1d>=1.4.0',
    'mamba_ssm',
])

if not wheel_ok:
    print("Prebuilt wheel not available. Falling back to source build (this can take 10-30+ minutes)...")
    build_env = {
        "MAX_JOBS": "2",  # safer on Kaggle memory during ninja build
        "TORCH_CUDA_ARCH_LIST": "7.5",
        "CAUSAL_CONV1D_FORCE_BUILD": "TRUE",
        "MAMBA_FORCE_BUILD": "TRUE",
    }
    run([
        sys.executable, '-m', 'pip', 'install',
        '--no-build-isolation',
        'causal_conv1d>=1.4.0',
        'mamba_ssm',
    ], extra_env=build_env)

# Quick import check so failures show early in setup cell.
run([sys.executable, '-c', 'import causal_conv1d, mamba_ssm; print("mamba kernels import OK")'])

src_path = str(REPO_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Penting: subprocess baru (python -m ...) tidak mewarisi sys.path kernel,
# jadi PYTHONPATH harus dipasang agar import f5_tts selalu berhasil.
prev_pythonpath = os.environ.get('PYTHONPATH', '')
os.environ['PYTHONPATH'] = src_path + (os.pathsep + prev_pythonpath if prev_pythonpath else '')

print('Dependency setup selesai.')
print('PYTHONPATH:', os.environ['PYTHONPATH'])


## Step 3 - GPU/CUDA sanity check

Notebook ini menargetkan multi-GPU T4x2.


In [ ]:
import torch
import accelerate

print('torch version      :', torch.__version__)
print('torch cuda version :', torch.version.cuda)
print('accelerate version :', accelerate.__version__)
print('cuda available     :', torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError('GPU tidak terdeteksi. Aktifkan GPU Accelerator di Kaggle settings.')

gpu_count = torch.cuda.device_count()
print('gpu count          :', gpu_count)
for i in range(gpu_count):
    print(f'  gpu[{i}]          :', torch.cuda.get_device_name(i))

if gpu_count < 2:
    print('WARNING: GPU terdeteksi < 2. Multi-GPU T4x2 tidak aktif di sesi ini.')
else:
    print('OK: Multi-GPU siap dipakai.')


## Step 4 - WANDB_API_KEY from Kaggle Secret

Buat secret bernama `WANDB_API_KEY` di Kaggle (Add-ons -> Secrets).


In [ ]:
import importlib

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('WANDB_SILENT', 'true')
os.environ.setdefault('WANDB__SERVICE_WAIT', '300')
os.environ.setdefault('WANDB_DIR', str(REPO_ROOT / 'wandb'))

WANDB_ENABLED = False

try:
    wandb = importlib.import_module('wandb')
    kaggle_secrets = importlib.import_module('kaggle_secrets')
    UserSecretsClient = getattr(kaggle_secrets, 'UserSecretsClient')

    secret_client = UserSecretsClient()
    wandb_api_key = secret_client.get_secret('WANDB_API_KEY')
    if wandb_api_key:
        os.environ['WANDB_API_KEY'] = wandb_api_key
        wandb.login(key=wandb_api_key, relogin=True)
        WANDB_ENABLED = True
except Exception as e:
    print('WANDB secret tidak tersedia / gagal dibaca:', e)

if not WANDB_ENABLED:
    os.environ['WANDB_MODE'] = 'offline'

print('WANDB_ENABLED:', WANDB_ENABLED)
print('WANDB_MODE   :', os.getenv('WANDB_MODE', 'online'))


## Step 5 - Convert metadata to absolute path format

`prepare_csv_wavs.py` mengharuskan `audio_file` path absolut.


In [ ]:
# Build combined dataset list from multiple metadata candidates
rows = []
processed_files = []

for metadata_name in METADATA_CANDIDATES:
    meta_path = DATA_ROOT / metadata_name
    if not meta_path.exists():
        continue
        
    processed_files.append(meta_path.name)
        
    # Auto-detect CSV delimiter from the header line (| for KCV, , for IndSP)
    with meta_path.open('r', encoding='utf-8-sig') as _probe:
        _header_line = _probe.readline()
    delimiter = '|' if '|' in _header_line else ','
    print(f'Detected metadata: {meta_path.name}  (delimiter={repr(delimiter)})')

    with meta_path.open('r', encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f, delimiter=delimiter)
        fieldnames = {(name or '').strip() for name in (reader.fieldnames or [])}
        expected_fields = {'audio_file', 'text'}

        if fieldnames != expected_fields:
            print(f"WARNING: Skipping {meta_path.name} - Header CSV harus memiliki kolom audio_file dan text.")
            continue

        for raw_row in reader:
            row = {(k or '').strip(): (v or '') for k, v in raw_row.items()}

            rel_audio = row.get('audio_file', '').strip()
            if not rel_audio:
                continue

            rel_path = Path(rel_audio)
            
            # Jika path sudah absolute, pakai langsung; jika relatif, cari di semua WAVS_DIR_CANDIDATES
            abs_audio = rel_path
            if not rel_path.is_absolute():
                found = False
                for wav_dir in WAVS_DIR_CANDIDATES:
                    candidate_audio = DATA_ROOT / wav_dir / rel_path.name
                    if candidate_audio.exists():
                        abs_audio = candidate_audio
                        found = True
                        break
                
                if not found:
                    # Fallback ke path gabungan default
                    abs_audio = DATA_ROOT / rel_path
                    
            abs_audio = abs_audio.resolve()

            rows.append({
                'audio_file': abs_audio.as_posix(),
                'text': row.get('text', '').strip(),
            })

if not rows:
    raise ValueError("Tidak ada data valid ditemukan dari file metadata.")

ABS_METADATA.parent.mkdir(parents=True, exist_ok=True)
with ABS_METADATA.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['audio_file', 'text'], delimiter='|')
    writer.writeheader()
    writer.writerows(rows)

print('ABS_METADATA:', ABS_METADATA)
print('Total Metadata merged :', processed_files)
print('Total Rows written    :', len(rows))


## Step 6 - Build F5-TTS training dataset (`raw.arrow`, `duration.json`, `vocab.txt`)


In [ ]:
workers = str(min(8, max(1, (os.cpu_count() or 2) // 2)))
prepare_cmd = [
    sys.executable,
    '-m',
    'f5_tts.train.datasets.prepare_csv_wavs',
    str(ABS_METADATA),
    str(PREPARED_DATA_DIR),
    '--pretrain',
    '--workers', workers,
    # FIX #6: pass tokenizer explicitly so vocab is built with the correct
    # scheme. Without this the script may default to 'pinyin' (Mandarin),
    # which produces garbage tokens for non-Mandarin (e.g. Indonesian) text.
    '--tokenizer', TOKENIZER,
]
run(prepare_cmd)

print('\nPrepared files:')
for p in sorted(PREPARED_DATA_DIR.glob('*')):
    print('-', p.name, '|', p.stat().st_size, 'bytes')


## Step 7 - Smoke test dataset loading


In [ ]:
from f5_tts.model.dataset import load_dataset

dataset = load_dataset(DATASET_NAME, TOKENIZER)
print('Loaded dataset type:', type(dataset).__name__)
print('Dataset length     :', len(dataset))


## Step 8 - Write Kaggle-specific Mamba3 config

Config ini dibuat otomatis agar konsisten dengan dataset kamu.


In [ ]:
config_name = 'kaggle_indonesian_mamba3.yaml'
config_path = REPO_ROOT / 'src' / 'f5_tts' / 'configs' / config_name
logger_yaml = 'wandb' if WANDB_ENABLED else 'null'

config_text = f"""
hydra:
  run:
    dir: ckpts/${{model.name}}_${{model.mel_spec.mel_spec_type}}_${{model.tokenizer}}_${{datasets.name}}/${{now:%Y-%m-%d}}/${{now:%H-%M-%S}}

datasets:
  name: {DATASET_NAME}
  batch_size_per_gpu: 1600
  batch_size_type: frame
  max_samples: 64
  num_workers: 2

optim:
  epochs: 50
  learning_rate: 2e-4
  num_warmup_updates: 4000
  grad_accumulation_steps: 2
  max_grad_norm: 1.0
  bnb_optimizer: False
  # FIX #7: bf16 lebih stabil dari fp16 untuk SSM scan panjang.
  # fp16 rentan overflow pada sequence panjang; bf16 memiliki range yang lebih
  # besar (sama dengan fp32) dan T4 Kaggle mendukung keduanya.
  mixed_precision: bf16

model:
  name: Mamba3TTS_Base
  tokenizer: {TOKENIZER}
  tokenizer_path: null
  backbone: Mamba3Backbone
  arch:
    dim: 1024
    depth: 22
    ff_mult: 2
    text_dim: 512
    text_mask_padding: True
    conv_layers: 4
    checkpoint_activations: False
    dropout: 0.0
    d_state: 128
    headdim: 64
    ngroups: 1
    rope_fraction: 0.5
    is_mimo: False
    mimo_rank: 1
    chunk_size: 64
  mel_spec:
    target_sample_rate: 24000
    n_mel_channels: 100
    hop_length: 256
    win_length: 1024
    n_fft: 1024
    mel_spec_type: vocos
  vocoder:
    is_local: False
    local_path: null

ckpts:
  logger: {logger_yaml}
  wandb_project: Mamba3-TTS
  wandb_run_name: ${{model.name}}_${{model.mel_spec.mel_spec_type}}_${{model.tokenizer}}_${{datasets.name}}
  wandb_resume_id: null
  log_samples: false
  save_per_updates: 10000
  keep_last_n_checkpoints: 3
  last_per_updates: 2000
  save_dir: ckpts/${{model.name}}_${{model.mel_spec.mel_spec_type}}_${{model.tokenizer}}_${{datasets.name}}
""".strip() + "\n"

config_path.write_text(config_text, encoding='utf-8')
print('Training config written to:', config_path)
print(config_path.read_text(encoding='utf-8').splitlines()[:12])


## Step 9 - Write Accelerate config for T4x2 (DDP)


In [ ]:
num_gpus = torch.cuda.device_count()
num_processes = 2 if num_gpus >= 2 else 1
distributed_type = 'MULTI_GPU' if num_processes > 1 else 'NO'

accelerate_cfg_path = REPO_ROOT / 'accelerate_kaggle.yaml'
accelerate_cfg_text = f"""
compute_environment: LOCAL_MACHINE
debug: false
distributed_type: {distributed_type}
downcast_bf16: 'no'
enable_cpu_affinity: false
machine_rank: 0
main_training_function: main
mixed_precision: bf16
num_machines: 1
num_processes: {num_processes}
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false
""".strip() + "\n"

accelerate_cfg_path.write_text(accelerate_cfg_text, encoding='utf-8')
print('Accelerate config:', accelerate_cfg_path)
print(accelerate_cfg_path.read_text(encoding='utf-8'))


## Step 10 - Launch multi-GPU training

Cell ini akan memulai training. Jalankan saat semua step sebelumnya sudah sukses.


In [ ]:
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

if torch.cuda.device_count() >= 2:
    os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

train_cmd = [
    sys.executable,
    '-m', 'accelerate.commands.launch',
    '--config_file', str(accelerate_cfg_path),
    'src/f5_tts/train/train.py',
    '--config-name', config_name,
]
run(train_cmd)


## Step 11 - Resume training (optional)

Jika sesi Kaggle putus, jalankan lagi cell launch training.
Trainer otomatis membaca checkpoint terbaru di folder `ckpts/...`.

Tips stabilitas:
- Jika OOM: turunkan `datasets.batch_size_per_gpu` (contoh: 1200 atau 800).
- Jika data loader error: turunkan `datasets.num_workers` ke `0`.
- Jika W&B secret tidak ada: training tetap jalan mode offline.
